In [16]:
import numpy as np

tag = "Ey6.000Vum_N100_L300"
Mx, My, Mz = (np.load(f"M{a}_{tag}.npy") for a in "xyz")

muB = 5.788e-5                      # eV/T
B = 0.1                             # T
pauli = [np.array([[0, 1], [1, 0]]),
         np.array([[0, -1j], [1j, 0]]),
         np.diag([1, -1])]

# full g-tensor:  M_a = (muB/2) sum_b g_ab sigma_b  ->  g_ab = Tr(M_a sigma_b) / muB
g = np.array([[np.trace(M @ s).real / muB for s in pauli] for M in (Mx, My, Mz)])

# g-factor for B along x, y, z (= Zeeman splitting / (muB B))
g_axis = np.linalg.norm(g, axis=1)

# principal values (sign of det g fixes the overall sign of the product)
g_principal = np.linalg.svd(g, compute_uv=False)
sign_det = np.sign(np.linalg.det(g))

print(f"g-tensor ({tag}), rows = B direction (x, y, z), cols = pseudospin (x, y, z):")
print(np.array2string(g, precision=5, suppress_small=True))
print(f"\ng_xx = {g_axis[0]:.5f}\ng_yy = {g_axis[1]:.5f}\ng_zz = {g_axis[2]:.5f}")
print(f"\nprincipal |g| = {np.round(g_principal, 5)},  sign(det g) = {sign_det:+.0f}")

# check: splitting from H_Z = B . M at B = 1 T along each axis
for a, M in zip("xyz", (Mx, My, Mz)):
    dE = np.ptp(np.linalg.eigvalsh(M))          # eV at 1 T
    print(f"B || {a}, 0.1 T: splitting = {B *dE*1e6:9.3f} ueV  ->  g = {dE/muB:.5f}")

g-tensor (Ey6.000Vum_N100_L300), rows = B direction (x, y, z), cols = pseudospin (x, y, z):
[[ 1.20517  0.       0.     ]
 [-0.      -3.31993 -0.00019]
 [-0.      -0.      13.60007]]

g_xx = 1.20517
g_yy = 3.31993
g_zz = 13.60007

principal |g| = [13.60007  3.31993  1.20517],  sign(det g) = -1
B || x, 0.1 T: splitting =     6.975 ueV  ->  g = 1.20517
B || y, 0.1 T: splitting =    19.216 ueV  ->  g = 3.31993
B || z, 0.1 T: splitting =    78.717 ueV  ->  g = 13.60007
